*Two agents, 112 identical games, and the two ways of reading the result that disagree.*

**You have two versions of an agent and you want to know which is better. You run both against the same set of opponents and count the wins.** One wins 108 of 112, the other 107. Same pool, same opponents. They look like the same agent, and no amount of squinting at 108 against 107 will separate them.

**The two are not close.** Compare them game by game instead of totalling first, and one of them is behind in **83 of those same 112 games**, by a median of \$7,656, at a p-value of 0.0000003. Nothing changed except the order of two operations: subtract first, then total, rather than total first and then subtract.

**Both agents are provided in this notebook as complete, runnable submission files**, so you can take them, run them against your own harness, or submit them. This is that comparison worked end to end on a real pair, with the data attached so you can re-run it and change the question. Both agents in it are exact replays of recorded routes with no decision layer, which keeps the example about the method. The short version of what it costs to get this wrong: **the version that looks equal on win rate is the one you would have shipped.**

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import comb
from pathlib import Path
from scipy import stats

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 130,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.6,
    "font.size": 9.5, "axes.titlesize": 11.5, "axes.titleweight": "bold",
    "axes.labelsize": 9.5, "legend.frameon": False,
    "lines.linewidth": 1.1, "lines.markersize": 4.0,
    "lines.solid_capstyle": "round", "axes.linewidth": 0.7,
    "xtick.major.width": 0.7, "ytick.major.width": 0.7, "patch.linewidth": 0.7,
})
INK, TEAL, AMBER, GREY, RED = "#0F172A", "#0F766E", "#B45309", "#94A3B8", "#B91C1C"

def find(name):
    for root in ("/kaggle/input", ".", "..", "data"):
        r = Path(root)
        if r.exists():
            for p in r.rglob(name):
                return p
    return None

WP = find("worked_example_paired.json")
if WP is None:
    raise FileNotFoundError(
        "worked_example_paired.json not found. Attach the dataset "
        "'destbreso/kaggriculture-benchmark-matchups' with Add Input, on the "
        "right of the editor, and make sure the version you attach is the "
        "latest one.")
W = json.loads(Path(WP).read_text())
G = pd.DataFrame(W["games"])
print(W["what"])
print()
for k in ("agent_A", "agent_B"):
    a = W[k]
    print(f"  {k[-1]}: an exact replay of {a['source']}, "
          f"episode {a['episode']}, seat {a['seat']}")
print(f"\n  {len(G)} games, {G.opponent.nunique()} distinct opponents, "
      f"seats {int((G.seat==0).sum())}/{int((G.seat==1).sum())}")

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; THE SETUP: What the two agents are, and why they were chosen this way</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p>Both are <b>pure replays</b>. Each one emits, at every turn, the action a recorded top-10 team actually took in one of its own episodes, reconstructed from the public episode API. Neither decides anything, neither reads the board, and neither contains a line of strategy. That is deliberate: an example about a measurement method should not depend on believing anything about the agents being measured.</p>
<ul>
<li><b>A</b> replays a route recorded by <b>Utkarsh #2</b>, ranked 2 at capture with a rating of 3,086.1.</li>
<li><b>B</b> replays a route recorded by <b>Mohamed abdelrazik</b>, ranked 9 at capture.</li>
</ul>
<p>Both routes are the <b>medoid</b> episode of that team's recorded games: the single real episode whose plan is most similar to all their others, so it is the most representative game they actually played rather than an average of several, which would produce a plan nobody ran.</p>
<p>The opponents are 33 distinct recorded teams, every one rated above 2,000, none of them either donor. Every game is replayed at its own resolved seed, from its own seat, against that opponent's own recorded 720 actions, so the games are reproducible to the dollar.</p>
<p><b>Why replays rather than something cleverer.</b> Two identical-looking win rates and a decisive paired difference is a property of the comparison, not of the agents. Using replays makes that checkable: there is no hidden state, no randomness, and nothing about either agent that could explain the result away.</p>

</div>
</details>

---

## The two agents, as code you can run

Both are the same fifteen lines with a different route fed into them. That is the point of the design and it is worth seeing before the numbers: **the route is an argument and the agent is the decision procedure**, which here decides nothing at all, so what you get is a pure replay.

In [ ]:
R = json.loads(Path(find("worked_example_routes.json")).read_text())

def make_replay_agent(route):
    '''Turn a recorded route into a player. This is the whole agent.'''
    PASS = {"farmer": ["PASS"], "hands": [], "market": []}

    def agent(obs):
        # DERIVE THE TURN, never read obs["step"]. Seat 1 receives a trimmed
        # observation with no `step` field: on the route below, 719 times out
        # of 719. Filling that absence with a default replays turn 0 forever.
        turn = int(obs["day"]) * 24 + int(obs["hour"])
        idx = turn + 1                    # a replay stores turn t at steps[t+1]
        if idx >= len(route):
            return dict(PASS)
        act = route[idx]
        if not isinstance(act, dict):
            return dict(PASS)
        return {"farmer": act.get("farmer") or ["PASS"],
                "hands": list(act.get("hands") or []),
                "market": list(act.get("market") or [])}
    return agent

agent_A = make_replay_agent(R["A"]["route"])
agent_B = make_replay_agent(R["B"]["route"])

rows = []
for k in ("A", "B"):
    r = R[k]
    rows.append({"agent": k, "donor": r["donor"], "rank": r["rank_at_capture"],
                 "rating at capture": r["rating_at_capture"], "turns": r["turns"],
                 "plan aligned": r["plan_aligned"],
                 "mutable turns": f"{r['mutable_turns']} ({r['mutable_share']})",
                 "medoid": r["medoid_similarity"]})
display(pd.DataFrame(rows))
for k in ("A", "B"):
    print(f"  {k}: {R[k]['source_field']}   sha256 {R[k]['sha256'][:16]}...")

### Where these came from, in the language this competition has grown

Neither route was written. Both were **recovered** from the competition's own episode API, which publishes every ranked game in full, and both are credited to the team that played them.

**A donor** is a team whose recorded games a route is taken from. Choosing one is not the same as choosing the strongest team, and the table above is why.

**The medoid** is the single recorded episode whose plan is most similar to all the others that team played. It is the most representative real game rather than an average, because an average of streams is a plan nobody ever ran and the engine may simply refuse it. Both routes here sit above 0.98, meaning these teams play very nearly the same game every time.

**The plan** is the construction: what gets hired, bought, planted and built. Both donors hold theirs across better than four episodes in five.

**The mutable surface** is the set of turns at which the donor itself varied across its own episodes. It is evidence that the game tolerates a different choice there, and it is where a decision layer would be licensed to deviate. The two differ by a factor of four, 142 turns against 35, which made A the more promising donor before a single game was played.

**Lineage 7.R** is the label for a pure replay that decides nothing. Both of these are 7.R. A route carrying a decision layer is a different animal and takes a different letter, because the two fail in different ways and comparing them without the distinction is how a project ends up with a table that cannot be read.

*The measurement behind the mutable surface, and how to tell which part of somebody's stream is safe to change, is in [Mutants at the top](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay).*

## Take the agents with you

The cell below writes two complete, self-contained submission files, one per agent, with the route compressed inside. No dataset, no imports beyond the standard library, nothing else to fetch. Download one, submit it, or run it against your own harness offline.

They are **not tuned and not ours**. Each is somebody else's recorded game played back, credited inside the file, and they are here because a worked example about measurement is more useful when you can run the things being measured.

In [ ]:
import base64, hashlib, zlib

HEAD = [
 "# Kaggriculture agent: a pure replay of a recorded route. Lineage 7.R.",
 "#",
 "# ATTRIBUTION. The behaviour in this file is NOT ours and is not recovered",
 "# source. It is the observable action stream of a game played by {donor},",
 "# ranked {rank} at capture, taken from the competition's own public episode",
 "# API:",
 "#     {source}",
 "#     route sha256 {sha}",
 "#",
 "# WHAT IT DOES. It emits, at every turn, the action that team took at that",
 "# turn. It reads nothing, decides nothing, adapts to nothing.",
 "#",
 "# THE ONE TRAP. The turn is derived from day and hour and NEVER from",
 "# obs['step'], which is absent for seat 1 in a trimmed observation: on this",
 "# route, 719 times out of 719. Filling that absence with a default replays",
 "# turn 0 for the whole game, silently, and the agent still finishes every",
 "# episode looking perfectly healthy.",
 "import base64, json, zlib",
 "",
 "_BLOB = (",
]
TAIL = [
 ")",
 "ROUTE = json.loads(zlib.decompress(base64.b85decode(_BLOB)))",
 "PASS = {'farmer': ['PASS'], 'hands': [], 'market': []}",
 "",
 "",
 "def agent(obs):",
 "    turn = int(obs['day']) * 24 + int(obs['hour'])",
 "    idx = turn + 1",
 "    if idx >= len(ROUTE):",
 "        return dict(PASS)",
 "    act = ROUTE[idx]",
 "    if not isinstance(act, dict):",
 "        return dict(PASS)",
 "    return {'farmer': act.get('farmer') or ['PASS'],",
 "            'hands': list(act.get('hands') or []),",
 "            'market': list(act.get('market') or [])}",
 "",
]

for k in ('A', 'B'):
    r = R[k]
    raw = json.dumps(r['route'], separators=(',', ':')).encode()
    blob = base64.b85encode(zlib.compress(raw, 9)).decode()
    head = [h.format(donor=r['donor'], rank=r['rank_at_capture'],
                     source=r['source_field'],
                     sha=hashlib.sha256(raw).hexdigest()) for h in HEAD]
    body = ['    "%s"' % blob[i:i+100] for i in range(0, len(blob), 100)]
    src = "\n".join(head + body + TAIL)
    name = 'agent_%s_replay.py' % k
    Path(name).write_text(src)
    print('  wrote %-22s %6.1f kB   replays %s' % (name, len(src)/1024, r['donor']))

print()
print('  Standard library only. Drop one into a submission and it runs.')
print()
print('  ' + '-'*66)
print('  agent_A_replay.py, first lines and the shape of the rest:')
print('  ' + '-'*66)
preview = Path('agent_A_replay.py').read_text().splitlines()
for line in preview[:20]:
    print('  ' + line)
print('  ' + preview[20][:74] + ' ...')
print(f'  ... {len(preview)-24} more lines of route, then:')
for line in preview[-14:]:
    print('  ' + line)

A word on why these are worth having even though neither is strong enough to be worth submitting on its own.

A replay is the **cheapest baseline that is not a strawman**. It plays a real game a real competitor really played, at a real rating, so a change you make has something honest to be measured against. Most offline harnesses compare a candidate to a hand-written baseline nobody would ever submit, and beating something bad is not evidence of being good.

They are also the **exact control** for anything built on top. Add a decision layer to one of these, switch the layer off, and you must get this file back byte for byte. That check is worth more than it sounds: it is the difference between measuring your change and measuring your change plus a bug.

# The first way: count the wins

The obvious comparison. Run both against the same pool, count how often each beat its opponent, put an interval on it.

In [ ]:
G["A_margin"] = G.A_bank - G.A_opponent_bank
G["B_margin"] = G.B_bank - G.B_opponent_bank

def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z*z/n
    c = p + z*z/(2*n)
    h = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (c-h)/d, (c+h)/d

rows = []
for name in ("A", "B"):
    w = int((G[f"{name}_margin"] > 0).sum())
    lo, hi = wilson(w, len(G))
    rows.append({"agent": name, "wins": f"{w}/{len(G)}",
                 "win rate": f"{100*w/len(G):.1f} %",
                 "95 % interval": f"[{100*lo:.1f} %, {100*hi:.1f} %]",
                 "median bank": f"{G[f'{name}_bank'].median():,.0f}"})
display(pd.DataFrame(rows))

wa = int((G.A_margin > 0).sum()); wb = int((G.B_margin > 0).sum())
tab = [[wa, len(G)-wa], [wb, len(G)-wb]]
print(f"Fisher exact on the two win counts: p = {stats.fisher_exact(tab)[1]:.3f}")
print("On this evidence the two agents are indistinguishable.")

**108 against 107.** The intervals sit almost on top of each other and a test on the two counts does not come close to rejecting. If this were the whole analysis the conclusion would be that the two agents are the same, and the next move would be to run more games.

More games would not have helped, and the reason is visible in the table above rather than in the statistics. **Both agents win almost every game.** A win rate is a proportion bounded at 1, and at 96 % it has almost no room left to move: it can separate an agent that wins from one that loses, and it cannot separate two agents that both win. The measurement is against its ceiling.

The information did not vanish. It is in *by how much*.

In [ ]:
print(f"  median margin, agent A   ${G.A_margin.median():>12,.0f}")
print(f"  median margin, agent B   ${G.B_margin.median():>12,.0f}")
print(f"  B wins by {G.B_margin.median()/G.A_margin.median():.2f} times as much, "
      f"on the same games")

**B beats its opponents by 66 % more money than A does**, on the same games, while both win at the same rate. Every one of those dollars was recorded and then thrown away by counting wins.

# The second way: subtract first

Every game was played by both agents on the same seed, from the same seat, against the same recorded opponent. So each game carries two margins, and their difference measures the agents with the game divided out.

In [ ]:
G["paired"] = G.A_margin - G.B_margin
up = int((G.paired > 0).sum())
dn = int((G.paired < 0).sum())

def sign_test(up, dn):
    n = up + dn
    k = max(up, dn)
    return min(1.0, 2 * sum(comb(n, j) for j in range(k, n+1)) / 2**n)

print(f"  A ahead in       {up:>4} games")
print(f"  A behind in      {dn:>4} games")
print(f"  ties             {len(G)-up-dn:>4}")
print(f"  median of the paired difference   {G.paired.median():>+12,.0f}")
print(f"  sign test                         p = {sign_test(up, dn):.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.2))

ax = axes[0]
bins = np.linspace(min(G.A_margin.min(), G.B_margin.min()),
                   max(G.A_margin.max(), G.B_margin.max()), 34)
ax.hist(G.A_margin, bins=bins, color=TEAL, alpha=0.35, lw=0, label="agent A")
ax.hist(G.B_margin, bins=bins, histtype="step", color=AMBER, lw=1.2,
        label="agent B")
ax.axvline(0, color=GREY, lw=0.9)
ax.annotate("both are almost entirely\non the winning side of zero,\nwhich is why the win rates tie",
            xy=(0, ax.get_ylim()[1]*0.62), xytext=(14, 0),
            textcoords="offset points", fontsize=8.5, color=INK, va="center")
ax.set_xlabel("margin against the opponent, dollars"); ax.set_ylabel("games")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.set_title("Counted separately: a tie against the ceiling", loc="left")
ax.legend(fontsize=8.5, loc="upper left")

ax = axes[1]
ax.hist(G.paired, bins=34, color=RED, alpha=0.35, lw=0)
ax.axvline(0, color=INK, lw=1.2)
ax.axvline(G.paired.median(), color=RED, lw=1.1, ls=(0, (4, 3)))
ax.annotate(f"median {G.paired.median():+,.0f}",
            xy=(G.paired.median(), ax.get_ylim()[1]*0.88), xytext=(-8, 0),
            textcoords="offset points", fontsize=9, color=RED, ha="right")
ax.set_xlabel("A minus B, on the same game, dollars"); ax.set_ylabel("games")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.set_title(f"Differenced game by game: A behind in {dn} of {len(G)}", loc="left")
plt.tight_layout(); plt.show()

Same 112 games in both panels. On the left, two piles that both sit almost entirely to the right of zero, which is the tie. On the right, the difference between them, sitting almost entirely to the left of it.

# What each step contributed

Two separate things happened, and it is worth keeping them apart because they are worth different amounts.

In [ ]:
sa, sb = G.A_margin.std(ddof=1), G.B_margin.std(ddof=1)
sp, rho, n = G.paired.std(ddof=1), G.A_margin.corr(G.B_margin), len(G)
se_unpaired = np.sqrt(sa**2/n + sb**2/n)
se_paired = sp/np.sqrt(n)

print(f"  spread of A's margins    ${sa:>12,.0f}")
print(f"  spread of B's margins    ${sb:>12,.0f}")
print(f"  correlation between them  {rho:>+12.3f}")
print(f"  spread of the difference ${sp:>12,.0f}   "
      f"(algebra predicts ${np.sqrt(sa**2+sb**2-2*rho*sa*sb):,.0f})")
print()
print(f"  standard error on the mean difference")
print(f"    if the two runs were independent   ${se_unpaired:>10,.0f}")
print(f"    paired on the same games           ${se_paired:>10,.0f}")
print(f"    pairing is worth                    {se_unpaired/se_paired:>10.2f}x")
print()
# A ratio on the standard error converts to games by squaring it, because the
# error falls with the square root of the sample. Quote the games figure rather
# than the ratio, since the ratio invites being read as a saving in games.
print(f"  in games, which is what you actually spend:")
print(f"    200 independent games each carry as much information")
print(f"    as {200 * (se_paired/se_unpaired)**2:.0f} paired ones")

**Switching from wins to margins is what made the difference visible at all**, because the win rate was against its ceiling and carried almost nothing. **Pairing is worth a further 1.47x** on the standard error, which is a real saving and a much smaller one than the first step.

In games rather than in standard errors that is 1.47 squared, so two hundred independent games buy what ninety-two paired ones do. Worth converting before quoting, because a ratio on the error reads as a bigger saving than it is.

That order is worth remembering, because the usual advice is stated the other way round. Pairing is the famous trick; choosing a statistic that is not saturated is the one that mattered here.

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; THE ALGEBRA BEHIND IT: Common random numbers, which is the oldest trick in simulation</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p>The two margins are not independent. Both agents played the <i>same</i> seed, so they saw the same weather, the same shop draws and the same opponent. Whatever made a game easy made it easy for both.</p>
<p>For any two quantities with spreads $s_A$ and $s_B$ and correlation $\rho$,</p>
$$\operatorname{Var}(A - B) = s_A^2 + s_B^2 - 2\rho\, s_A s_B$$
<p>so a positive $\rho$ subtracts. It is not an approximation and it does not depend on any distribution being normal. All of the season's luck that both agents shared is removed by the subtraction, and what is left is the part that is actually them.</p>
<p>This is <b>common random numbers</b>, standard in simulation since the 1950s and one of the most reliable variance reductions available anywhere. The condition it needs is the one people skip: the pairing has to be <i>exact</i>. Same seed, same seat, same opponent, same everything you are not measuring. Two independent runs of 200 games each are not paired, and no amount of averaging turns them into a pair.</p>
<p>A second, smaller point hides in the choice of test. The paired differences here have heavy tails, so the sign test is used rather than a t-test: it asks only which side of zero each difference lands on, and a handful of blowouts cannot steer it. Ties are excluded, because a sign test is defined on the non-zero differences and this game produces real ties.</p>
<ul>
<li>Law, A. M. & Kelton, W. D. <i>Simulation Modeling and Analysis.</i> McGraw-Hill.</li>
<li>Glasserman, P. & Yao, D. D. (1992). Some guidelines and guarantees for common random numbers. <i>Management Science</i> 38(6), 884-908.</li>
<li>Wilcoxon, F. (1945). Individual comparisons by ranking methods. <i>Biometrics Bulletin</i> 1(6), 80-83.</li>
</ul>

</div>
</details>

---

## What that buys, in games

The useful way to price a variance reduction is to ask how many games each method would need to reach the same confidence.

In [ ]:
def sign_power(n, p, alpha=0.05):
    lo = stats.binom.ppf(alpha/2, n, 0.5) - 1
    hi = stats.binom.ppf(1-alpha/2, n, 0.5) + 1
    return float(stats.binom.cdf(lo, n, p) + stats.binom.sf(hi-1, n, p))

def n_for(p, target=0.80):
    return next((n for n in range(10, 20000) if sign_power(n, p) >= target), None)

obs = dn / (up + dn)
print(f"  observed paired rate: {100*obs:.1f} % of games favour one side")
print(f"  games needed to establish that at 80 % power: {n_for(obs)}")
print()
rows = []
for p in (0.55, 0.60, 0.65, 0.70, 0.74):
    n = n_for(p)
    rows.append({"a true paired rate of": f"{100*p:.0f} %",
                 "needs this many paired games": n,
                 "at 1.13 s a game": f"{2*n*1.13/60:.0f} min"})
display(pd.DataFrame(rows))

## Could you fix it with more games instead?

Yes, and it is worth pricing before deciding, because "run more games" is the reflex and here it is the expensive answer.

In [ ]:
pa = float((G.A_margin > 0).mean())
pb = float((G.B_margin > 0).mean())
p_ = (pa + pb) / 2
za, zb = 1.96, 0.84                       # 5 % two-sided, 80 % power
n_wins = ((za*np.sqrt(2*p_*(1-p_)) + zb*np.sqrt(pa*(1-pa) + pb*(1-pb)))
          / (pa - pb))**2
discordant = int(((G.A_margin > 0) != (G.B_margin > 0)).sum())

print(f"  the two win rates differ by {100*(pa-pb):.2f} percentage points")
print(f"  games PER AGENT to establish that by counting wins: {n_wins:,.0f}")
print(f"  at 1.13 s a game, for both agents: {2*n_wins*1.13/3600:.1f} hours")
print()
print(f"  pairing does not rescue it either: only {discordant} of {len(G)} games")
print(f"  are discordant, one agent winning where the other lost, and a test on")
print(f"  {discordant} discordant pairs has no power at all")
print()
print(f"  switching to the margin instead: 35 paired games,")
print(f"  the same conclusion, {n_wins/35:,.0f} times cheaper")

**Seven and a half thousand games against thirty-five.** More games does work eventually, because the interval on a proportion keeps narrowing. It is simply the worst trade available: two hundred times the compute to recover information that was in the data all along and was discarded at the moment of counting.

Pairing alone does not save the win rate either. Only **9 of the 112 games** are discordant, meaning one agent won where the other lost; in the rest the two agree. A paired test on nine observations has no power, so the pairing has nothing to work with. **Pairing helps a statistic that varies. It cannot help one that has stopped varying.**

There is a third fix, and it is the one to reach for when the statistic is not yours to choose. **The pool is too easy for both agents.** Their opponents bank a median of 70,114 against A's 80,330 and B's 95,869, so most of these games were decided before they started. A pool that included opponents which genuinely beat these two would move the win rate off its ceiling and give it room to discriminate again. That is a pool-design fix rather than a statistical one, and it costs a rebuild rather than compute.

Which one you use depends on what you control. If you still have the margins, change the statistic and pay nothing. If all you kept was a win column, rebuild the pool. If neither, budget the hours honestly.

### And here is that answer measured rather than calculated

The arithmetic above says more games works and costs 217 times as much. The same two agents were also run over **421** identical games instead of 112, which is enough to watch the saturation come apart in practice.

In [ ]:
BIG = pd.DataFrame(json.loads(Path(find("worked_example_paired_421.json")).read_text())["games"])
BIG["A_margin"] = BIG.A_bank - BIG.A_opponent_bank
BIG["B_margin"] = BIG.B_bank - BIG.B_opponent_bank
BIG["paired"] = BIG.A_margin - BIG.B_margin

rows = []
for label, df in (("112 games", G), ("421 games", BIG)):
    wa = int((df.A_margin > 0).sum()); wb = int((df.B_margin > 0).sum())
    u = int((df.paired > 0).sum()); d = int((df.paired < 0).sum())
    disc = int(((df.A_margin > 0) != (df.B_margin > 0)).sum())
    rows.append({"sample": label,
                 "A wins": f"{wa}/{len(df)}", "B wins": f"{wb}/{len(df)}",
                 "gap in win rate": f"{100*(wa-wb)/len(df):+.1f} pts",
                 "discordant games": disc,
                 "paired, A behind in": f"{d}/{u+d}",
                 "sign test p": f"{sign_test(u, d):.1e}"})
display(pd.DataFrame(rows))

Both readings move, and they move for different reasons.

**The paired test gains thirteen orders of magnitude** on the p-value, which is the ordinary reward for four times the sample and is not surprising.

**The win rate stops being saturated**, which is. At 112 games it read 108 against 107, a gap of under one point that nothing could act on. At 421 it reads 387 against 409, and the discordant games, the ones where one agent won and the other lost, go from **9 to 42**. There is now something for a paired test on the win indicator to work with, where before there was not.

So the honest answer to "can more games fix a saturated statistic" is **yes, and you need roughly four times as many before it begins to show and two hundred times as many before it is established**, while changing the statistic sees it immediately. Which one is available depends on whether you kept the margins.

One thing does not move at all, and it is the reason any of this was worth doing: **the direction is the same in both samples, and it was already correct at 112.** The paired comparison had the answer four times cheaper and never changed its mind.

# What happened when this was put on the ladder

*Added 2026-08-17. The section above was written before any of it was submitted.*

The argument so far is that a paired difference sees things a win count cannot. That is a claim about a method, and a method claim is worth what its out-of-sample test is worth. So two agents were **submitted for the purpose**, at the same minute, and left to play.

They are not the exact pair above. Agent A is one of them; the other is a third route, recorded by **Masataka Suzuki**, ranked 8 at capture, chosen the same way and equally not mine. That substitution is not a convenience: the ladder freezes a submission's rating when it stops receiving games, so comparing a live rating against a retired one compares two different moments of a moving field. **Two agents fielded in the same minute is the only version of this comparison that controls for that.**

**The prediction, written into the submission messages before either had played a game:** the Suzuki route finishes above the Utkarsh route, and both finish well below the agent they were measured against.

Offline, the paired test on 421 identical games said the two are separated by a median of \$621. The margin between two banks in this game has a standard deviation of about \$27,200, so **the predicted difference was one fortieth of the noise in a single game**. A win count over any affordable sample cannot see a difference that small. The sign test could: it reached p = 1.5e-13, not because the effect is large but because it is consistent.

### The result

| | offline, paired, 421 games | ladder, 2026-08-17 | episodes |
|---|---|---|---|
| Suzuki route | ahead in **286 of 421** | **2,003.8** | 88 |
| Utkarsh route | behind | **1,853.4** | 93 |

Started eighteen seconds apart, read at nearly the same exposure. **The ordering the paired test predicted is the ordering the ladder returned**, and the 150-point gap it returned came from a predicted difference forty times smaller than the noise of one game.

That is the part worth taking: a difference far below the resolution of any single observation was called offline and confirmed by an instrument with no connection to the one that called it.

### And the half that failed, which is the same size as the half that worked

The same offline harness that got the ordering right got the **level** badly wrong.

| | offline win rate | real ladder win rate | overstated by |
|---|---|---|---|
| Utkarsh route | 90.3 % | 59.1 % | 31 points |
| Suzuki route | 92.6 % | 68.2 % | 24 points |

Those are not small errors, and they are in the same direction both times. An earlier agent measured here at 72 % went on to win 13 % of its real games, so the gap has narrowed from 59 points to about 27 without closing.

**The two results together say something more precise than either alone.** A paired comparison cancels everything the two agents share, including whatever is unrepresentative about the pool they were both measured on. A win rate cancels nothing: it inherits every way the pool differs from the field, and this pool's 40 opponents collapse to nineteen distinct behaviours. So the pairing survives a bad pool and the level does not.

**Quote a paired difference. Do not quote a pool win rate as a forecast.** I did the second thing for a week and it cost a week.

### One more thing the ladder said, and it was not in the plan

Both agents played the same window from the same minute. They drew **82 and 88 distinct opponents, with 8 in common**.

Submitting two agents at once controls the calendar. It does not give them the same opponents, because matchmaking pairs on rating and the two separate almost immediately. So even this design, which is the strongest available, compares two agents against two nearly disjoint populations, and the ratings are trustworthy only because a rating is built to normalise for opponent strength while a raw win rate is not.

An earlier pair measured this way had their win rates point the *opposite* way to their ratings. This pair's point the same way. Nothing in a win rate tells you which of those two situations you are looking at.

# What to take from it

**Pair, then difference, then test.** The order matters more than the sample size. Two independent samples of 200 games each are worth about 92 paired ones for this question, which is the same statement as the 1.47x above squared.

**A win rate is a comparison against the pool, not against the other agent.** Two agents with the same win rate on the same pool can be far apart, and the sign of the difference is not recoverable from the two totals. If you only ever record win rates, that information was never stored.

**Use the median and the sign test, not the mean and a t-test.** The margins here have heavy tails and the mean of a heavy-tailed sample is steered by its rarest games.

**And if the two agents cannot be paired, say so rather than comparing anyway.** Two agents live on a rating-matched ladder do not meet the same opponents, so their win rates are measurements against different populations. Measured on two of ours over one window: 99 distinct opponents for one, 38 for the other, **zero in common**.

---

## Where the pieces come from

Each of these is one measurement this notebook leans on, rather than a general reading list.

- [**Know Your Noise**](https://www.kaggle.com/code/destbreso/kaggriculture-know-your-noise) is where the spread of a margin, the correlation between the two banks in a game, and the heavy tails that justify the sign test are derived and tested.
- [**Six checks before you waste a submission**](https://www.kaggle.com/code/destbreso/six-checks-before-you-waste-a-submission) puts this comparison in context as one of six, and covers how many paired games an effect of a given size needs.
- [**Kaggriculture Measure Your Agent**](https://www.kaggle.com/code/destbreso/kaggriculture-measure-your-agent) is the harness that runs all of it in a fixed order, and the two places it refuses to produce a number at all.
- [**How many opponents is a pool of forty?**](https://www.kaggle.com/code/destbreso/how-many-opponents-is-a-pool-of-forty) is why the pool in the denominator is smaller than it looks.
- [**Everyone is playing the same opening**](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening) is the mechanism behind that: unrelated teams emit identical actions deep into a game.
- [**Mutants at the top**](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay) is how the two routes used here were located, and which part of a recorded stream is stable enough to replay.

The games themselves, and every matchup they were drawn from, are in [the replayable benchmark](https://www.kaggle.com/datasets/destbreso/kaggriculture-benchmark-matchups), CC0.